# RQ7.1 — Model Compression: Seed 14

Compare the effects of quantisation, structured pruning and knowledge distillation on MobileNetV3-Small and EfficientNet-B0.

| Setting | Value |
|---|---|
| Training seed | 14 |
| Batch size | 64 |
| Classes | 38 |
| Train / validation / test | 38,147 / 5,448 / 10,704 |
| Baseline training | 3 head-training + 17 fine-tuning epochs |
| Pruning recovery | 5 epochs per candidate |
| CE and KD students | 20 epochs each |
| Source commit | `c9849c4fed12dd15e05801c860ae37af078e60a4` |

Run all sections with a GPU runtime. Training starts only after the source, environment, dataset and preflight checks succeed. Each execution uses a new timestamped output directory. This reproduction entrypoint uses the cleaned GitHub source. Historical Kaggle outputs retain their original source-bundle provenance. Dataset assignments, architectures and training budgets are unchanged; complete-run validation of this entrypoint is separate from those historical runs.


## 1. Configuration

Confirm the seed and image path. Set `RUN_FULL=True` to enable training after all checks pass. For private GitHub access, provide a read-access token through the `GITHUB_TOKEN` secret. Kaggle saved background runs require that secret to be attached in advance and Internet enabled. Never put credentials in code or outputs.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, shutil, subprocess, threading
IS_KAGGLE=Path('/kaggle/working').is_dir()
RUN_FULL=False
SEEDS=[14]
REPO_URL='https://github.com/vbhaga89/COS801_Project.git'
SOURCE_COMMIT='c9849c4fed12dd15e05801c860ae37af078e60a4'
assert len(SEEDS)==1 and SEEDS[0] in (14,27,42)
if IS_KAGGLE:
    IMAGE_SOURCE=Path('/kaggle/input/datasets/makungundlovu/cos801-rq7-1-audited-plantvillage-archive/rq71_plantvillage')
    BASE=Path('/kaggle/working')
    BACKUP_BASE=None
else:
    from google.colab import drive
    drive.mount('/content/drive')
    IMAGE_SOURCE=Path('/content/drive/MyDrive/COS801_data/rq71_plantvillage.tar')
    BASE=Path('/content')
    BACKUP_BASE=Path('/content/drive/MyDrive/COS801_Project/rq7.1-model-compression/results/colab_seed14')
assert IMAGE_SOURCE.exists(), f'Correct IMAGE_SOURCE: {IMAGE_SOURCE}'
ROOT=BASE/datetime.now(timezone.utc).strftime('rq71_seed14_%Y%m%dT%H%M%S_%fZ')
ROOT.mkdir(exist_ok=False)
BACKUP=BACKUP_BASE/ROOT.name if BACKUP_BASE else None
if BACKUP: BACKUP.mkdir(parents=True,exist_ok=False)
os.environ['PYTHONUNBUFFERED']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
print('Platform:', 'Kaggle' if IS_KAGGLE else 'Colab', 'Seeds:',SEEDS,flush=True)
print('Local output:',ROOT,'Persistent backup:',BACKUP,flush=True)


## 2. Pinned GitHub source

Check out the exact cleaned repository commit and validate the fixed manifest fingerprints. The checkout is detached: later branch updates do not change this run. No source ZIP is required.


In [ ]:
import base64
import getpass
token=None
if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('GITHUB_TOKEN')
    except Exception:
        pass
else:
    try:
        from google.colab import userdata
        token=userdata.get('GITHUB_TOKEN')
    except Exception:
        token=getpass.getpass('GitHub token for repository read access (hidden; blank for public access): ').strip() or None
checkout_env=dict(os.environ,GIT_TERMINAL_PROMPT='0')
if token:
    credential=base64.b64encode(('x-access-token:'+token).encode()).decode()
    checkout_env.update(GIT_CONFIG_COUNT='1',GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+credential)
    del credential
REPOSITORY=ROOT/'repository'
REPOSITORY.mkdir(exist_ok=False)
def git(args):
    result=subprocess.run(['git',*args],cwd=REPOSITORY,env=checkout_env,text=True,capture_output=True)
    if result.returncode:
        raise RuntimeError('Repository checkout failed. Check Internet access, SOURCE_COMMIT and read access. For a private repository, provide the GITHUB_TOKEN secret. Never paste a token into notebook code or output.')
    return result.stdout.strip()
try:
    git(['init','--quiet'])
    git(['remote','add','origin',REPO_URL])
    git(['fetch','--quiet','--depth','1','origin',SOURCE_COMMIT])
    git(['checkout','--quiet','--detach','FETCH_HEAD'])
    assert git(['rev-parse','HEAD'])==SOURCE_COMMIT, 'Unexpected source commit'
finally:
    checkout_env.clear()
    token=None
PROJECT=REPOSITORY/'rq7.1-model-compression'
assert (PROJECT/'stage_plantvillage.py').is_file()
assert (PROJECT/'splits'/'assignments.csv').is_file()
sys.path.insert(0,str(PROJECT))
from hybrid_protocol import validate
protocol,_=validate(PROJECT/'splits')
(ROOT/'source_provenance.json').write_text(json.dumps(dict(repository=REPO_URL,source_commit=SOURCE_COMMIT,seeds=SEEDS,manifest_directory='splits',split_protocol=protocol),indent=2))
def run(args):
    print('$',' '.join(map(str,args)),flush=True)
    subprocess.run(list(map(str,args)),cwd=PROJECT,check=True)
print('Verified GitHub source commit:',SOURCE_COMMIT,flush=True)
print('Verified final split counts:',protocol['counts'],flush=True)


## 3. Environment

Verify CUDA availability, install the scoped dependencies and record hardware and package versions. Retain the runtime's compatible PyTorch/torchvision pair. Run the protocol integration checks before data staging.


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU runtime required; stop and select one'
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
run([sys.executable,'-m','pip','install','-r',PROJECT/'requirements.txt'])
run([sys.executable,'-u','check_environment.py','--require-cuda','--output',ROOT/'environment.json'])
with (ROOT/'pip_freeze.txt').open('w') as handle:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=handle,check=True)
run([sys.executable,'-u','test_hybrid_integration.py'])
run([sys.executable,'-u','test_final_split_defaults.py'])


## 4. Dataset staging

Copy the image archive to runtime-local storage. Stage and hash-check all 54,299 manifest images according to the revised assignment table. Original Drive data remain unchanged; existing staging destinations are not overwritten.


In [ ]:
source=IMAGE_SOURCE
if not IS_KAGGLE and IMAGE_SOURCE.is_file():
    source=ROOT/IMAGE_SOURCE.name
    print('Copying image archive once from Drive...',flush=True)
    shutil.copyfile(IMAGE_SOURCE,source)
DATA=ROOT/'images'
MANIFESTS=PROJECT/'splits'
run([sys.executable,'-u','stage_plantvillage.py','--source',source,'--output',DATA,'--manifest-dir',MANIFESTS])


## 5. Preflight validation

Validate the reviewed manifests, split counts, image hashes and experiment configuration. This section does not start training.


In [ ]:
COMMAND=[sys.executable,'-u','run_three_seeds.py','--config',str(PROJECT/'configs/full.json'),'--data-dir',str(DATA),'--manifest-dir',str(MANIFESTS),'--output-root',str(ROOT/'results'),'--split-version','hybrid-v2','--seeds',*map(str,SEEDS)]
run(COMMAND)
PREFLIGHT_PASSED=True


## 6. Training and backup

This section runs after section 5 passes when executing the notebook from top to bottom. Do not rerun sections 1–4 in the same session.

The pipeline runs linear probes, both baselines, compression and selected-model analyses. Accuracy-gate rejections are retained as negative results. Unexpected errors stop execution.

Colab results are written under `/content/rq71_hybrid_v2_<timestamp>/results` and copied to `MyDrive/COS801_Project/rq7.1-model-compression/results/colab_seed14/<timestamp>` every five minutes during training and on normal exit. Backup does not implement checkpoint resume and cannot guarantee recovery after abrupt runtime loss. Verify the final backup before ending the session.


In [ ]:
if not RUN_FULL:
    print('Training disabled. Preflight completed; no models trained.',flush=True)
else:
    assert PREFLIGHT_PASSED
    def backup_snapshot():
        if BACKUP is None: return
        for parent in (ROOT/'results', ROOT):
            files=parent.rglob('*') if parent.name=='results' else parent.glob('*')
            for file in files:
                if not file.is_file() or (parent==ROOT and file.suffix not in ('.json','.txt','.log')): continue
                before=file.stat()
                destination=BACKUP/file.relative_to(ROOT)
                if destination.exists() and destination.stat().st_size==before.st_size and destination.stat().st_mtime_ns>=before.st_mtime_ns: continue
                destination.parent.mkdir(parents=True,exist_ok=True)
                temporary=destination.with_name(destination.name+'.copying')
                shutil.copy2(file,temporary)
                after=file.stat()
                if (before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns): temporary.replace(destination)
    stop=threading.Event()
    def periodic_backup():
        while not stop.wait(300):
            try:
                backup_snapshot()
                print('[backup] snapshot completed',flush=True)
            except Exception as exc: print('[backup] warning:',exc,flush=True)
    worker=threading.Thread(target=periodic_backup,daemon=True) if BACKUP else None
    if worker: worker.start()
    try:
        with (ROOT/'launcher.log').open('w') as log:
            with subprocess.Popen(COMMAND+['--execute'],cwd=PROJECT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
                for line in process.stdout:
                    print(line,end='',flush=True); log.write(line); log.flush()
                code=process.wait()
        if code: raise RuntimeError(f'Launcher exit {code}; preserve logs and inspect before retrying')
    finally:
        stop.set()
        if worker: worker.join(timeout=30)
        if not worker or not worker.is_alive(): backup_snapshot()
        else: print('Backup still active; verify before closing',flush=True)


## 7. Results and figures

Inspect each run manifest and final comparison table before interpreting results. The complete figure index includes training histories and any generated calibration, risk–coverage and compression plots. Choose figure indices in `SHOW_FIGURES` to display them.


In [ ]:
import pandas as pd
from IPython.display import display, Image
for path in sorted((ROOT/'results').glob('run_*/seed_*/full_run_manifest.json')):
    print(path.relative_to(ROOT)); display(json.loads(path.read_text()))
for path in sorted((ROOT/'results').glob('run_*/seed_*/final_comparison.csv')):
    print(path.relative_to(ROOT)); display(pd.read_csv(path))
FIGURES=sorted((ROOT/'results').rglob('*.png'))
for index,path in enumerate(FIGURES): print(index,path.relative_to(ROOT))
print('Figure count:',len(FIGURES),'Local results:',ROOT/'results','Drive backup:',BACKUP)


In [ ]:
SHOW_FIGURES=[]
for index in SHOW_FIGURES:
    print(FIGURES[index].relative_to(ROOT))
    display(Image(filename=str(FIGURES[index])))


## 8. Reproducibility and limitations

- Aggregate seed 14 with seeds 27, 42 only after checking source, split and configuration fingerprints and software environments.
- Compare deployment latency on a common CPU target; hosted CPU measurements are proxies, not low-cost-device validation.
- Known author leaf IDs and exact image duplicates are disjoint across partitions. Unresolved leaf identities and unscreened near duplicates remain limitations.
- Test membership is fixed. Historical test results were inspected before the validation-protocol revision; disclose this and do not pool legacy and hybrid results as equivalent repetitions.
- Review both models' stage statuses, accuracy-gate outcomes, tables, histories, analysis artifacts and backups before declaring the experiment complete.

- Historical saved cloud runs used the original frozen bundle. This notebook uses the pinned cleaned repository commit; report its source provenance separately.
